In [1]:
from os import listdir
from os.path import join, isdir

# browse through all folders and get all file paths
def get_files(dir):
    files = []
    while dir:
        file = dir.pop(0)

        if isdir(file):
            dir.extend([join(file,i) for i in listdir(file)])
            files.extend(get_files(dir))
        else:
            files.append(file)
    return files

folder = ['Testfolder']
files = get_files(folder)

In [2]:
files

['Testfolder/SOP IT Validation Guideline Doc ID 10000011139_v07.pdf',
 'Testfolder/SOP Equipment Qualification and Production Process Validation_V04.pdf',
 'Testfolder/General-Principles-of-Software-Validation---Final-Guidance-for-Industry-and-FDA-Staff.pdf',
 'Testfolder/SOP Development of measurement systems draft v0.4.docx',
 'Testfolder/HTML/I-0188_NTT-VMP.html',
 'Testfolder/PDF/I-0188_NTT-VMP.pdf',
 'Testfolder/Markdown/I-0188_NTT-VMP.md']

In [3]:
from tika import parser
from docx import Document
import keybert

/home/tobias/anaconda3/envs/cas_main_ta/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [4]:
# my own list of stopwords for use in the keyBERT model
my_stopwords = [
    'schott',
    'schott pharma',
    'approver',
    'creator',
    'releaser'
]

In [ ]:
# Identify keywords per file with keyBERT
keywords = {}
for i in files:
    raw = ""

    if i.endswith('.pdf'):
        parsed_pdf = parser.from_file(i)
        raw = parsed_pdf['content'] if parsed_pdf and 'content' in parsed_pdf else ""

    elif i.endswith('.docx'):
        doc = Document(i)
        raw = "\n".join([p.text for p in doc.paragraphs])
    
    if not raw.strip():
        continue

    model = keybert.KeyBERT()
    new_keywords = model.extract_keywords(raw,keyphrase_ngram_range=(1,3), stop_words=['english']+my_stopwords,top_n=8)
    new_keywords.sort(key=lambda x: x[1], reverse=True)
    keywords[i]=(new_keywords)


/home/tobias/anaconda3/envs/cas_main_ta/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['pharma'] not in stop_words.
  warnings.warn(
/home/tobias/anaconda3/envs/cas_main_ta/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['pharma'] not in stop_words.
  warnings.warn(
/home/tobias/anaconda3/envs/cas_main_ta/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['pharma'] not in stop_words.
  warnings.warn(
/home/tobias/anaconda3/envs/cas_main_ta/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing

In [6]:
keywords

{'Testfolder/SOP IT Validation Guideline Doc ID 10000011139_v07.pdf': [('pharma new validation',
   0.5868),
  ('validation within pharma', 0.5822),
  ('software documentation pharma', 0.5374),
  ('validation 04 software', 0.5338),
  ('specification released pharma', 0.5251),
  ('documentation pharma', 0.5231),
  ('of printing pharma', 0.5182),
  ('technical documents pharma', 0.5164)],
 'Testfolder/SOP Equipment Qualification and Production Process Validation_V04.pdf': [('sop qualification validation',
   0.6092),
  ('tpl_template_sop_for_p standard doc', 0.537),
  ('qualification validation procedure', 0.5296),
  ('validation qualification documentation', 0.5286),
  ('sop equipment qualification', 0.526),
  ('specifications templates sops', 0.5251),
  ('tpl_template_sop_for_p standard', 0.5237),
  ('to sop validation', 0.5221)],
 'Testfolder/General-Principles-of-Software-Validation---Final-Guidance-for-Industry-and-FDA-Staff.pdf': [('fda validation guidance',
   0.6969),
  ('fda sta

In [7]:
# Make lists of lists into one big list
def flatten(lists: list) -> list:
    result = []
    for el in lists:
        if not isinstance(el, str):
            result.extend(flatten(el))
        else:
            result.append(el)
    return result

In [8]:
# Create sets containing all keywords per file
for i in keywords:
    flat_i = flatten([j[0] for j in keywords[i]])

    flat_i_split = []
    for j in flat_i:
        flat_i_split.extend(j.split())

    kwset = set(flat_i_split)
    print(i, kwset)

Testfolder/SOP IT Validation Guideline Doc ID 10000011139_v07.pdf {'04', 'new', 'printing', 'specification', 'technical', 'software', 'of', 'validation', 'pharma', 'documents', 'documentation', 'within', 'released'}
Testfolder/SOP Equipment Qualification and Production Process Validation_V04.pdf {'templates', 'sop', 'procedure', 'qualification', 'specifications', 'to', 'standard', 'validation', 'sops', 'equipment', 'documentation', 'doc', 'tpl_template_sop_for_p'}
Testfolder/General-Principles-of-Software-Validation---Final-Guidance-for-Industry-and-FDA-Staff.pdf {'specification', 'guidance', 'applicable', 'of', 'fda', 'in', 'validation', 'staff', 'other'}
Testfolder/SOP Development of measurement systems draft v0.4.docx {'computerized', 'for', 'cosmetical', 'in', 'equipment', 'inspection', 'measurement', 'vision', 'engineering', 'system'}
Testfolder/PDF/I-0188_NTT-VMP.pdf {'operational', 'plan', 'qr', 'e3', 'qualification', 'kontron', 'rd', 'oqprr', 'protocol', 'validation', 'qp', 're

In [9]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.petersen_graph()
subax1 = plt.subplot(121)
nx.draw(G, with_labels=True, font_weight='bold')
subax2 = plt.subplot(122)
nx.draw_shell(G, nlist=[range(5, 10), range(5)], with_labels=True, font_weight='bold')

ModuleNotFoundError: No module named 'matplotlib'